In [ ]:
from math import log2, sqrt
import numpy as np
import scipy
import matplotlib.pyplot as plt
from scipy.signal import spectrogram
from scipy.fft import fft

from torch.nn.functional import conv1d
import torch.nn as nn
from torch.fft import fft, ifft
import torch

from einops import rearrange, repeat

# Develop band plan of time vs frequency
# Tune to band
# While still on this band
    # get 200ms of samples
    # Use polyphase filter to channelize to 100 channels
        # reshape input
        # apply filters to each channel
        # reshape
        # Do ifft
    # shift into per-channel buffer
    # Do 2.4e4 pt fft per channel
    # Estimate noise floor
    # Check if peak is above threshold
    # save channel to file if true
    
# Fs = 2.4e6
# NSAMPLES = int(Fs)
# TOVERLAP = 0.2
# NOVERLAP = int(TOVERLAP*Fs)
# x = np.random.rand(NSAMPLES) + 1j*np.random.rand(NSAMPLES)

# %timeit X = fft(x, NSAMPLES)

class PolyphaseChannelizer(nn.Module):
    def __init__(self, M: torch.Tensor, h: torch.Tensor):
        """
        Takes input x of complex samples and channelizes it into M channels with a polyphase filter constructed from h
        Initialization:
            - M The number of channels. Must be a power of 2.
            - h The prototype filter. Number of taps must be integer multiple of M.
        Inputs:
            - x The input signal(s). [B] x T, where B is batches and T is time (samples). T must be an integer multiple of M.
            - prev The last M/len(h)-1 samples of the previous correlation. [B] x M/len(h)-1
        Outputs:
            - The channelized samples [B] x M x T/M
            - The last M/len(h)-1 samples of the correlation. [B] x M/len(h)-1
        """
        super().__init__()
        assert float(int(log2(M))) == log2(M), f"M ({M}) must be a power of 2"
        assert h.size(-1) % M == 0, f"Size of filter ({h.size(-1)}) must be integer multiple of number of channels ({M})"
        self.M = M
        self.filter_bank = rearrange(h, "(T M) -> M 1 T", M=M)
        self.filter_bank = self.filter_bank.flip(-1)
        self.taps = self.filter_bank.size(-1)

    def init_prev(self):
        return torch.zeros(2, self.M, self.taps-1)

    def forward(self, x, prev):
        assert prev.size(-1) == self.taps-1, f"The last dimension of prev ({prev.size(-1)}) must be taps-1 ({self.taps-1})"
        assert prev.size(-2) == self.M, f"The second to last dimension of prev ({prev.size(-2)}) must be M ({self.M})"
        assert x.dtype in [torch.complex32, torch.complex64, torch.complex128], "x must be complex"   

        # assert prev.size()[0:-1] == x.size()[0:-1], f"The optional batch and IQ dimensions must match between x {x.size()[0:-1]} and prev {prev.size()[0:-1]}"
        assert x.size(-1) % self.M == 0
        if x.ndim == 1:
            dummy_batch = True
            x = x.unsqueeze(0)
        
        # Shift input to left in freq domain by 1/2 channel. This helps the output channels all line up right
        t = torch.arange(0, x.size(-1))
        x = x*torch.exp(-1j*2*torch.pi*t/(self.M*2))

        # Reshape into decimated streams with I/Q dimension collapsed into the batch so we can apply same filter to both
        x_down = rearrange(torch.view_as_real(x.resolve_conj()), "B (T M) IQ -> (B IQ) M T", M = self.M)

        # Do full convolution
        x_filt_temp = conv1d(x_down, self.filter_bank, groups=self.M, padding=self.taps-1)

        # Separate out parts where it only partially overlapped
        x_filt_pre = x_filt_temp[...,:self.taps-1]
        x_filt = x_filt_temp[...,self.taps-1:-(self.taps-1)]
        x_filt_post = x_filt_temp[...,-(self.taps-1):]

        # Assemble the full convolution for this frame
        x_filt = torch.cat([x_filt_pre + prev, x_filt], dim=-1)

        # Perform ifft across channels
        x_filt = torch.view_as_complex(rearrange(x_filt, "(B IQ) M T -> B M T IQ", IQ=2).contiguous())
        x_chan = ifft(x_filt, dim=-2, norm="forward")
        y = x_chan
        
        # Rearrange output channels
        y = torch.roll(y.flip(dims=(-2,)), 1, dims=-2)
        # circ shift in frequency domain by half channel.
        t = torch.arange(0, y.size(-1))
        y = y*torch.exp(-1j*2*torch.pi*t/2)
        # Now fft(y).flatten() ~= fft(x)

        if dummy_batch:
            y = y.squeeze(0) # Remove batch if just a dummy dim

        return y, x_filt_post

class Spectrogramer(nn.Module):
    def __init__(self, n_fft: int, hop_length=None):
        super().__init__()
        self.n_fft = n_fft
        if hop_length is None:
            hop_length = n_fft // 4
        self.hop_length = hop_length
        self.overlap_frames = (self.n_fft // self.hop_length) - 1
        self.overlap = self.n_fft - self.hop_length
        self.window = torch.hann_window(n_fft, periodic=True)

    def forward(self, x: torch.Tensor, prev: torch.Tensor):
        assert prev.size(-1) == self.overlap, f"last dimension of prev must be n_fft - hop_length = {self.overlap}"
        x = torch.cat([prev, x], dim=-1)
        y = torch.stft(x, self.n_fft, self.hop_length, window=self.window, center=False)
        y = y[:-self.overlap_frames] # discard partial frames
        prev = x[..., -self.overlap:]

        return y, prev
    
    def init_prev(self):
        return torch.zeros(self.overlap)

def gen_samps(N, M, Fs):
    """Generate 1/2 channel bandwidth BPSK signals centered in M channels with ascending amplitude"""
    samps_per_sym = M*4
    x = 2 * torch.randint(0,2,(M, N//samps_per_sym)) - 1
    x = repeat(x, "M S -> M (S rep)", rep=samps_per_sym)
    t = torch.arange(0, N).unsqueeze(0)
    f = torch.arange(start=Fs/M/2, step=Fs/M, end=Fs).unsqueeze(-1)
    shifts = torch.exp(1j*2*torch.pi*f*t/Fs)
    x_shifted = x * shifts
    x_shifted = x_shifted * torch.linspace(start=0.3, end=1.0, steps = M).unsqueeze(-1)
    x = x_shifted.sum(0)
    return x

def get_samps(N, Fs):
    

In [ ]:
from scipy.signal import firwin

# Setup frames
FRAMES = 16
frame_s = 6.4
Fs = 2.048e6
N = Fs*frame_s

# Setup channelizer
M = 64
Fs_chan = Fs // M
N_chan = N//M
taps = 2048
Fc = int(Fs / M / 2)

h = torch.tensor(firwin(numtaps=taps, cutoff=Fc, window='blackmanharris', fs=Fs), dtype=torch.float32)
pc = PolyphaseChannelizer(M,h)
pc_prev = pc.init_prev()

n_stft = 1024
hop_length = n_stft//2
sg = Spectrogramer(n_stft, hop_length)
sg_prev = repeat(sg.init_prev(), "N -> (M N)", M=M)

for i in range(FRAMES):
    x = gen_samps(N,M,Fs)
    x_chan, pc_prev = pc(x, pc_prev)
    sgs, sg_prev = sg(x_chan, sg_prev)
    # Estimate noise floor
    # Get peaks
    # Threshold
    

X = fft(x)
plt.plot(10*X.abs().log10())
plt.show()



print(x_chan.size())

plt.plot(x_chan[4][0:100])
plt.show()

X_chan_mag = 10*fft(x_chan).abs().log10()
# f = torch.fft.fftfreq(int(N//M), 1/(Fs/M))
# X_chan_mag = 20*torch.log10(X_chan.abs())

plt.plot(X_chan_mag.flatten())
plt.show()

for chan in X_chan_mag[:4]:
    print(chan.size())
    plt.plot(chan)
    plt.show()

In [180]:
Fs = 32000
s = 6.4
N = s*Fs
NFFT = 1024
hop_length = NFFT/2

print("Time resolution: ", NFFT/Fs)
print("Freq resolution: ", Fs//NFFT)
print(f"Dimensions (px) txf:  {int(N/hop_length)} x {NFFT}")
print(f"Dimensions (ph) txf KHz:  {N/Fs} x {Fs*1e-3}")

Time resolution:  0.032
Freq resolution:  31
Dimensions (px) txf:  400 x 1024
Dimensions (ph) txf KHz:  6.4 x 32.0


In [ ]:
%timeit pc(x, prev)

In [ ]:
from rtlsdr import RtlSdr
sdr = RtlSdr()

# configure device
sdr.sample_rate = Fs
sdr.center_freq = 91.7e6
sdr.gain = 36

samples = sdr.read_samples(N)
sdr.close()

x = torch.tensor(samples, dtype=torch.complex64)
x_chan, pc_prev = pc(x, pc_prev)

plt.plot(20*torch.log10(fft(x).abs()))
plt.show()

f = torch.fft.fftfreq(int(N//M), 1/(Fs/M))
X_chan_mag = 20*torch.log10(X_chan.abs())

for chan in X_chan_mag[30:35]:
    print(chan.size())
    plt.plot(f, chan)
    plt.show()

In [ ]:
# M=5
# T=30
# IQ=2
# H=3
# x = torch.arange(0,60, step=0.2)
# xr = rearrange(x, "(IQ T) -> 1 IQ T", IQ=2)
# xc = rearrange(xr, "B IQ (T M) -> (B IQ) M T", M=M)
# xf = rearrange(torch.arange(M*H).to(dtype=torch.float32), "(M T) -> M 1 T", M=M)
# xf = xf.flip(-1)
# print(xr)
# print(xf)
# print(xc)
# xcf = conv1d(xc, xf, groups=M, padding=H-1)
# print(xcf)
# print(xr.size())
# print(xc.size())
# print(xf.size())
# print(xcf.size())